<a href="https://colab.research.google.com/github/NaveenHerath/llm-from-scratch/blob/main/day3_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embeddings

First lets repeat what we did on day 2
- Tokenize the characters and cut those numbers into training chunkz

In [22]:
#1.Load the text
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
#this reads the shakespeare file into a one single string


In [23]:
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [24]:
len(text)

1115394

In [25]:
#2.Building the vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
#this sorts the unique charactes in the dataset and identify them

In [26]:
#3.Translators
stoi = {ch: i  for i, ch in enumerate(chars)} #string to integer
itos = {i: ch for i, ch in enumerate(chars)} #int to str

In [27]:
#4.Encode and Decode
encode = lambda s: [stoi[c] for c in s] #encoder
decode = lambda l: ''.join([itos[i] for i in l]) #decoder

In [28]:
encode("hello there")

[46, 43, 50, 50, 53, 1, 58, 46, 43, 56, 43]

In [29]:
encode("h")
#here the enocder we used is a simple numerater.
#each character into a unique number. not a complex vector.

[46]

In [30]:
#5.Turn the whole book into a tensor
import torch
data = torch.tensor(encode(text), dtype = torch.long)
#store those numbers in to a tensor

In [33]:
data.shape
#so this is tensor is a one dimensional array.

torch.Size([1115394])

In [34]:
#6.Train/Validation split
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
#so the one dimensional matrix/array is now partitioned into 2 sections

In [37]:
#7.Get batch
block_size = 8 #how many characters models sees at once
batch_size = 4 #how many chunks we process in parallel

#we are writing a function to run through the batches of blocks
def get_batch(split):
  data = train_data if split == 'train' else val_data #tells what part to look at
  ix = torch.randint(len(data) - block_size, (batch_size,)) #picks 4 random strating positions, starting position must be within 0 to (len-blocksize)
  xb = torch.stack([data[i:i+block_size] for i in ix]) #cuts out 8 characters starting at ach of those positions torch.ctack thenput those on top of eac otherr to make a 4*8 grid
  yb = torch.stack([data[i+1:i+block_size+1] for i in ix]) #this do the same thing shifted one text to the right to make them labels
  return xb, yb

#xb = x batch and yb =  y batch
#so we now we have xb, yb of 4*8 tensors which are consist of 4 random 8 character lines each while x has the random picks and the y has the labels.

Day 03 work

In [38]:
xb, yb = get_batch('train')
print(xb.shape)

#xb and yb are the tensors we created using block sizes and batches. input and label
#firstly we only use the xb,yb comes later

torch.Size([4, 8])


In [39]:
#A. make the token table
import torch.nn as nn

n_embd = 32 #num of weight you are giving for each individual character/ number of dimensions you specify
token_embedding_table = nn.Embedding(vocab_size, n_embd) #number of rows are equalent to num of character/vocab size
print(token_embedding_table.weight.shape)

torch.Size([65, 32])


In [42]:
#B. pass a batch through it
tok_emb = token_embedding_table(xb) #here we pas through the 4*8 xb through the 65*32 embedding table
print(xb.shape)
print(tok_emb.shape)
#so by doing that we connect the each character with their unique row in the embedding table,
#so every number got deeper from just one number to 32 number row with adjustable weights

torch.Size([4, 8])
torch.Size([4, 8, 32])
